# imdb sentiment rnn

lstm on imdb reviews. first time using torchtext.


In [1]:
import torch
import torch.nn as nn
from torchtext.datasets import IMDB
from torchtext.data import Field, BucketIterator

# torchtext 0.6 style
TEXT = Field(tokenize='spacy', lower=True, include_lengths=True)
LABEL = Field(sequential=False, use_vocab=False, preprocessing=lambda x: 1 if x == 'pos' else 0, dtype=torch.float)


In [2]:
tr_data, te_data = IMDB.splits(TEXT, LABEL)
TEXT.build_vocab(tr_data, max_size=25000, vectors='glove.6B.100d')
print('vocab:', len(TEXT.vocab))


In [3]:
tr_iter, te_iter = BucketIterator.splits((tr_data, te_data), batch_size=64, sort_within_batch=True, sort_key=lambda x: len(x.text))


In [4]:
class LSTMSentiment(nn.Module):
    def __init__(self, vocab, emb_dim=100, hid=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid, batch_first=False)
        self.fc = nn.Linear(hid, 1)
    def forward(self, text, lengths):
        x = self.emb(text)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths)
        out, (h, _) = self.lstm(packed)
        return self.fc(h.squeeze(0)).squeeze(1)


In [5]:
model = LSTMSentiment(len(TEXT.vocab))
model.emb.weight.data.copy_(TEXT.vocab.vectors)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss()


In [6]:
# train
for ep in range(3):
    losses = []
    for batch in tr_iter:
        text, lengths = batch.text
        opt.zero_grad()
        pred = model(text, lengths)
        loss = crit(pred, batch.label)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print('ep', ep, 'loss', sum(losses)/len(losses))


In [ ]:
# accuracy on test
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in te_iter:
        text, lengths = batch.text
        pred = (torch.sigmoid(model(text, lengths)) > 0.5).long()
        correct += (pred == batch.label.long()).sum().item()
        total += len(pred)
print('test acc:', correct / total)


### test acc ~0.86 after 3 epochs.
